# Modele 2 - Collaborative Filtering : Factorisation Matricielle ALS

Objectif : recommander 5 articles par utilisateur en exploitant les patterns collectifs d'interactions ("les utilisateurs qui ont lu les memes articles que toi ont aussi lu...").


## 1. Chargement des donnees


In [1]:
import pandas as pd
import numpy as np
import os
import json
import time
import joblib
from scipy.sparse import csr_matrix, save_npz

DATA_RAW = "../data/raw/"
DATA_PROC = "../data/processed/"
OUT = "../out/"
CF_DIR = os.path.join(OUT, "models", "collaborative")
CFG_DIR = os.path.join(OUT, "models", "config")

os.makedirs(CF_DIR, exist_ok=True)
os.makedirs(CFG_DIR, exist_ok=True)

In [2]:
# Donnees preparees par l'EDA
clicks = pd.read_parquet(os.path.join(DATA_PROC, "clicks_enriched.parquet"))
interactions = pd.read_parquet(os.path.join(DATA_PROC, "interactions.parquet"))
articles = pd.read_csv(os.path.join(DATA_RAW, "articles_metadata.csv"))

# Mappings
with open(os.path.join(DATA_PROC, "user_map.json"), "r") as f:
    user_map = {int(k): v for k, v in json.load(f).items()}
with open(os.path.join(DATA_PROC, "article_map.json"), "r") as f:
    article_map = {int(k): v for k, v in json.load(f).items()}

# Mappings inverses
user_map_inv = {v: k for k, v in user_map.items()}
article_map_inv = {v: k for k, v in article_map.items()}

print(f"Clicks : {len(clicks):,}")
print(f"Users : {len(user_map):,}")
print(f"Articles : {len(article_map):,}")

Clicks : 2,988,181
Users : 322,897
Articles : 46,033


## 2. Construction de la matrice d'interaction

La librairie `implicit` (versions recentes) attend une matrice creuse user-item (CSR). Les valeurs representent la confiance dans l'interaction (ici : nombre de clics par paire).


In [3]:
rows = interactions["user_id"].map(user_map).values
cols = interactions["click_article_id"].map(article_map).values
data = interactions["nb_clicks"].values.astype(np.float32)

n_users = len(user_map)
n_items = len(article_map)

user_item = csr_matrix((data, (rows, cols)), shape=(n_users, n_items))

print(f"Matrice user-item : {user_item.shape}")
print(f"Elements non-nuls : {user_item.nnz:,}")
print(f"Densite : {user_item.nnz / (n_users * n_items) * 100:.4f}%")

Matrice user-item : (322897, 46033)
Elements non-nuls : 2,950,710
Densite : 0.0199%


## 3. Entrainement du modele ALS


In [4]:
try:
    import implicit
    print(f"implicit version : {implicit.__version__}")
except ImportError:
    import subprocess
    subprocess.check_call(["pip", "install", "implicit", "--break-system-packages", "-q"])
    import implicit
    print(f"implicit installe, version : {implicit.__version__}")

from implicit.als import AlternatingLeastSquares

implicit version : 0.7.2


### 3.1 Recherche d'hyperparametres

On teste plusieurs configurations et on mesure le Hit Rate sur un echantillon.

Pour le split, on utilise un split temporel : le dernier article clique par chaque user = test.


In [5]:
# Split temporel
clicks_sorted = clicks.sort_values(["user_id", "click_timestamp"])
last_clicks = clicks_sorted.groupby("user_id").tail(1)
train_clicks = clicks_sorted.drop(last_clicks.index)

print(f"Train : {len(train_clicks):,} interactions")
print(f"Test  : {len(last_clicks):,} interactions")

# Matrice train
train_interactions = train_clicks.groupby(["user_id", "click_article_id"]).size().reset_index(name="nb_clicks")
rows_train = train_interactions["user_id"].map(user_map).values
cols_train = train_interactions["click_article_id"].map(article_map).values
data_train = train_interactions["nb_clicks"].values.astype(np.float32)
user_item_train = csr_matrix((data_train, (rows_train, cols_train)), shape=(n_users, n_items))

# Ground truth test
test_ground_truth = dict(zip(last_clicks["user_id"], last_clicks["click_article_id"]))
print(f"Ground truth : {len(test_ground_truth):,} paires")

Train : 2,665,284 interactions
Test  : 322,897 interactions
Ground truth : 322,897 paires


In [6]:
configs = [
    {"factors": 50,  "regularization": 0.01, "iterations": 15},
    {"factors": 50,  "regularization": 0.1,  "iterations": 15},
    {"factors": 100, "regularization": 0.01, "iterations": 15},
    {"factors": 100, "regularization": 0.1,  "iterations": 15},
    {"factors": 100, "regularization": 0.1,  "iterations": 30},
    {"factors": 150, "regularization": 0.1,  "iterations": 20},
]

results = []

# Echantillon pour evaluation rapide
sample_size = min(5000, len(test_ground_truth))
sample_users = np.random.RandomState(42).choice(
    list(test_ground_truth.keys()), size=sample_size, replace=False
)

for cfg in configs:
    print(f"Training : factors={cfg['factors']}, reg={cfg['regularization']}, iter={cfg['iterations']}...", end=" ")
    
    model = AlternatingLeastSquares(
        factors=cfg["factors"],
        regularization=cfg["regularization"],
        iterations=cfg["iterations"],
        random_state=42,
        use_gpu=False
    )
    
    t0 = time.time()
    model.fit(user_item_train)
    train_time = time.time() - t0
    
    hits_at_5 = 0
    for uid in sample_users:
        true_article = test_ground_truth[uid]
        if true_article not in article_map:
            continue
        uid_mapped = user_map.get(uid)
        if uid_mapped is None:
            continue
        
        rec_ids, rec_scores = model.recommend(
            uid_mapped, user_item_train[uid_mapped], N=5, filter_already_liked_items=True
        )
        
        true_mapped = article_map[true_article]
        if true_mapped in rec_ids[:5]:
            hits_at_5 += 1
    
    hr5 = hits_at_5 / sample_size
    results.append({**cfg, "hit_rate": hr5, "train_time": train_time})
    print(f"HR={hr5:.4f} | {train_time:.1f}s")

results_df = pd.DataFrame(results)
results_df

Training : factors=50, reg=0.01, iter=15... 

/home/ui/proj10/.env/lib/python3.11/site-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 16 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/15 [00:00<?, ?it/s]

HR=0.1468 | 19.0s
Training : factors=50, reg=0.1, iter=15... 

  0%|          | 0/15 [00:00<?, ?it/s]

...

  0%|          | 0/30 [00:00<?, ?it/s]

HR=0.1314 | 43.2s
Training : factors=150, reg=0.1, iter=20... 

  0%|          | 0/20 [00:00<?, ?it/s]

HR=0.1104 | 31.1s


,factors,regularization,iterations,hit_rate,train_time
0,50,0.01,15,0.1468,18.995383
1,50,0.10,15,0.1466,20.242376
2,100,0.01,15,0.1322,22.191527
3,100,0.10,15,0.1310,20.421311
4,100,0.10,30,0.1314,43.227249
5,150,0.10,20,0.1104,31.122402


### 3.2 Entrainement du meilleur modele sur donnees completes


In [7]:
best_cfg = results_df.loc[results_df["hit_rate"].idxmax()]
print(f"Meilleure config : factors={int(best_cfg['factors'])}, reg={best_cfg['regularization']}, iter={int(best_cfg['iterations'])}")
print(f"Hit Rate : {best_cfg['hit_rate']:.4f}")

best_model = AlternatingLeastSquares(
    factors=int(best_cfg["factors"]),
    regularization=best_cfg["regularization"],
    iterations=int(best_cfg["iterations"]),
    random_state=42,
    use_gpu=False
)

t0 = time.time()
best_model.fit(user_item)
print(f"Entrainement donnees completes : {time.time() - t0:.1f}s")
print(f"User factors : {best_model.user_factors.shape}")
print(f"Item factors : {best_model.item_factors.shape}")

Meilleure config : factors=50, reg=0.01, iter=15
Hit Rate : 0.1468


  0%|          | 0/15 [00:00<?, ?it/s]

Entrainement donnees completes : 19.5s
User factors : (322897, 50)
Item factors : (46033, 50)


## 4. Test des recommandations


In [8]:
def recommend_als(model, user_id, user_item_matrix, user_map, article_map_inv, top_n=5):
    uid_mapped = user_map.get(user_id)
    if uid_mapped is None:
        return []
    rec_ids, rec_scores = model.recommend(
        uid_mapped, user_item_matrix[uid_mapped], N=top_n, filter_already_liked_items=True
    )
    recs = []
    for idx, score in zip(rec_ids, rec_scores):
        article_id = article_map_inv.get(int(idx))
        if article_id is not None:
            recs.append({"article_id": article_id, "score": round(float(score), 4)})
    return recs

test_users = [0, 1, 5, 100, 1000]
for uid in test_users:
    recs = recommend_als(best_model, uid, user_item, user_map, article_map_inv, top_n=5)
    n_read = len(clicks[clicks["user_id"] == uid]["click_article_id"].unique())
    print(f"User {uid} | Articles lus : {n_read} | Recommandations :")
    for r in recs:
        meta = articles[articles["article_id"] == r["article_id"]]
        cat = meta["category_id"].values[0] if len(meta) > 0 else "?"
        print(f"  article_id={r['article_id']:>6d} | score={r['score']:.4f} | cat={cat}")
    print()

User 0 | Articles lus : 8 | Recommandations :
  article_id=293114 | score=0.2571 | cat=421
  article_id=284985 | score=0.2112 | cat=412
  article_id=272218 | score=0.1316 | cat=399
  article_id=160940 | score=0.1055 | cat=281
  article_id=124749 | score=0.1042 | cat=250

User 1 | Articles lus : 12 | Recommandations :
  article_id=284463 | score=0.4747 | cat=412
  article_id=207122 | score=0.4657 | cat=331
  article_id=119592 | score=0.1681 | cat=247
  article_id=293301 | score=0.1368 | cat=421
  article_id=336430 | score=0.1215 | cat=437

User 5 | Articles lus : 84 | Recommandations :
  article_id=161506 | score=0.3985 | cat=281
  article_id=160474 | score=0.3528 | cat=281
  article_id= 39894 | score=0.3350 | cat=66
  article_id= 64409 | score=0.3166 | cat=134
  article_id=352979 | score=0.3034 | cat=442

User 100 | Articles lus : 10 | Recommandations :
  article_id=276970 | score=0.1198 | cat=409
  article_id=293513 | score=0.1105 | cat=421
  article_id=207122 | score=0.1068 | cat=331

## 5. Benchmark temps d'inference


In [9]:
uid_test = 100
uid_mapped = user_map[uid_test]

times = []
for _ in range(100):
    t0 = time.time()
    best_model.recommend(uid_mapped, user_item[uid_mapped], N=5, filter_already_liked_items=True)
    times.append(time.time() - t0)

print(f"Temps moyen : {np.mean(times)*1000:.1f} ms")
print(f"Temps median : {np.median(times)*1000:.1f} ms")
print(f"Compatible Lambda (<1s) : {'Oui' if np.mean(times) < 1 else 'Non'}")

Temps moyen : 0.4 ms
Temps median : 0.2 ms
Compatible Lambda (<1s) : Oui


## 6. Sauvegarde des artefacts


In [ ]:
# Modele ALS
joblib.dump(best_model, os.path.join(CF_DIR, "als_model.joblib"))

# Matrice user-item
save_npz(os.path.join(CF_DIR, "user_item_matrix.npz"), user_item)

# Mappings
with open(os.path.join(CF_DIR, "user_map.json"), "w") as f:
    json.dump({str(k): v for k, v in user_map.items()}, f)
with open(os.path.join(CF_DIR, "article_map.json"), "w") as f:
    json.dump({str(k): v for k, v in article_map.items()}, f)
with open(os.path.join(CF_DIR, "article_map_inv.json"), "w") as f:
    json.dump({str(k): v for k, v in article_map_inv.items()}, f)

# Config
config = {
    "als_factors": int(best_cfg["factors"]),
    "als_regularization": float(best_cfg["regularization"]),
    "als_iterations": int(best_cfg["iterations"]),
    "n_users": n_users,
    "n_items": n_items
}
with open(os.path.join(CFG_DIR, "model_config.json"), "w") as f:
    json.dump(config, f, indent=2)

# Sauvegarde le ground truth pour le notebook 4
with open(os.path.join(CF_DIR, "test_ground_truth.json"), "w") as f:
    json.dump({str(k): v for k, v in test_ground_truth.items()}, f)

print("Artefacts sauvegardes :")
for root, dirs, files in os.walk(CF_DIR):
    for f in sorted(files):
        fpath = os.path.join(root, f)
        size = os.path.getsize(fpath) / 1e6
        rel = os.path.relpath(fpath, OUT)
        print(f"  {rel} : {size:.1f} Mo")

Artefacts sauvegardes :
  models/collaborative/als_model.joblib : 73.8 Mo
  models/collaborative/als_model.npz : 73.8 Mo
  models/collaborative/article_map.json : 0.8 Mo
  models/collaborative/article_map_inv.json : 0.8 Mo
  models/collaborative/test_ground_truth.json : 5.6 Mo
  models/collaborative/user_item_matrix.npz : 5.5 Mo
  models/collaborative/user_map.json : 5.6 Mo
